# Ximenez: Parse (sentence-level)

Produces sentence-level TOKEN, DOC, and DOCMAP tables for the Ximenez edition.
OHCO: `folio, side, para_num, sent_num, token_num`

Source: `../../textos/xml/xom-all-flat-mod-pnums.xml`

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
src_id = 'ximenez'
xml_path = '../../textos/xml/xom-all-flat-mod-pnums.xml'

## XML to LINE

In [ ]:
xml_lines = open(xml_path).readlines()

In [ ]:
els = dict(
    lang = '',
    folio = 0,
    side = 0,
    para_num = 0,
    lb = 0,
)

data = []
ana_list = []

for line in xml_lines:

    if re.match(r"<div xml:lang", line):
        els['lang'] = line.split('"')[1].split('"')[0]
        els['para_num'] = 0

    if re.match(r"^<pb ", line):
        f, s = line.split("xom-")[1].split('"')[0].split('-')
        els['folio'] = int(f[1:])
        els['side'] = int(s[1:])
        els['lb'] = 0

    if re.match(r"^<p ", line):
        els['para_num'] += 1

    if re.match(r"^<lb n=", line):
        els['lb'] += 1
        els['lb_str'] = ' '.join(line.split("/>")[1:]).strip()

        ana_num = 0
        rs = line.split("<rs ")
        if len(rs) > 1:
            for r in rs[1:]:
                ana = r.split('"')[1]
                ana_num += 1
                ana_list.append((els['lang'], els['folio'], els['side'], els['para_num'], els['lb'], ana_num, ana))

        data.append(tuple(els.values()))

In [ ]:
LINE = pd.DataFrame(data, columns=els.keys())
LINE = LINE.set_index(['lang', 'folio', 'side', 'para_num', 'lb'])
assert LINE.index.has_duplicates == False, "LINE has duplicates"
LINE

## Clean line strings

In [ ]:
LINE['lb_str_plain'] = (
    LINE.lb_str
    .str.replace(r"<[^>]+/?>", "", regex=True)
    .str.replace(" –", "–", regex=False)
)
chars = {
    'Ꜩ': 'Tz',
    'ꜩ': 'tz',
    'ꜫ': "q'",
    'ÿ': 'i',
}
for char in chars:
    LINE.lb_str_plain = LINE.lb_str_plain.str.replace(char, chars[char], regex=False)
LINE

## LINE to PARA

Aggregate K'iche' lines into paragraphs. `folio` and `side` take the first value
in case a paragraph spans a manuscript page break.

In [ ]:
LINE_QUC = LINE.loc['quc'].reset_index()
PARA = (
    LINE_QUC
    .groupby('para_num')
    .agg(
        folio=('folio', 'first'),
        side=('side', 'first'),
        doc_str=('lb_str_plain', lambda x: ' '.join(map(str, x)))
    )
    .reset_index()
)
PARA.doc_str = PARA.doc_str.str.replace('– ', '', regex=False).str.strip()
PARA

## PARA to SENT

Split each paragraph into sentences on sentence-terminal punctuation.
`sent_num` is the position within the paragraph.

In [ ]:
SENT = (
    PARA
    .assign(doc_str=lambda df: df.doc_str.str.split(r'(?<=[.!?])\s+'))
    .explode('doc_str')
    .dropna(subset=['doc_str'])
)
SENT = SENT[SENT.doc_str.str.strip() != ''].copy()
SENT['sent_num'] = SENT.groupby('para_num').cumcount()
SENT = SENT.reset_index(drop=True)
SENT

## SENT to DOCRAW → DOC, DOCMAP

Sentence is the chosen DOC level. `doc_id` is a sequential integer index.

In [ ]:
DOCRAW = SENT[['folio', 'side', 'para_num', 'sent_num', 'doc_str']].copy()
DOCRAW.index.name = 'doc_id'

DOC = DOCRAW[['doc_str']]
DOCMAP = DOCRAW[['folio', 'side', 'para_num', 'sent_num']]

print(f"{len(DOC):,} sentences from {DOCMAP.para_num.nunique()} paragraphs")
DOC.head(10)

In [ ]:
DOCMAP.head(10)

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", "", regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f"{src_id}-TOKEN.csv")
DOC.to_csv(f"{src_id}-DOC.csv")
DOCMAP.to_csv(f"{src_id}-DOCMAP.csv")
print("Saved to notebooks/doc_tables/")